# Founder Fade Curve Predicts OSS Survival — Demo

This notebook demonstrates a pilot experiment implementing a **trajectory shape-descriptor pipeline** to predict OSS project survival after founder departure.

**Key question:** Does the shape of a founder's involvement trajectory over time predict whether their open-source project survives after they leave?

**What this demo shows:**
- Synthetic trajectory generation (smooth fade, abrupt cliff, plateau-then-cliff)
- Computation of trajectory shape descriptors (slope, cliff indicator, fade index)
- Synthetic validation of descriptor assertions
- Model comparison results from the full pilot (static features vs. shape descriptors vs. combined)
- Falsification control analysis

**Results summary:** Static features (AUC=0.857) and combined features (AUC=0.898) predict survival, but trajectory shape descriptors alone (AUC=0.408) do not significantly predict survival beyond static features.

In [ ]:
# Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

# Additional packages needed for this demo
_pip('scikit-learn==1.6.1')

In [ ]:
# Imports
import json
import os
import numpy as np
import pandas as pd
from scipy.stats import theilslopes
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# Data loading helper
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-ad55a2-founder-fade-curve-predicts-oss-survival/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# Load data
data = load_data()
examples = data['datasets'][0]['examples']
print(f"Loaded {len(examples)} examples")

## Configuration

Tunable parameters for the demo. These control the synthetic trajectory generation and validation.

In [ ]:
# Configuration
N_TRAJECTORIES = 3  # Number of synthetic trajectories per pattern (default: 10)
TRAJECTORY_LENGTH = 24  # Months per trajectory (default: 24)
FADE_LAMBDA = 0.08  # Exponential decay rate for smooth fade
CLIFF_MONTH = 18  # Month when abrupt cliff occurs
NOISE_LEVEL = 0.02  # Noise standard deviation
SEED = 42  # Random seed for reproducibility

## Phase 0: Synthetic Trajectory Generation

We generate three types of synthetic trajectories to validate our descriptor pipeline:
1. **Smooth fade**: Gradual exponential decay in founder involvement
2. **Abrupt cliff**: Sudden drop in involvement at a specific month
3. **Plateau-then-cliff**: Stable involvement followed by sharp decline

Let's generate and visualize these trajectories.

In [ ]:
# Synthetic trajectory generators (from method.py)
def gen_smooth_fade(n=TRAJECTORY_LENGTH, lam=FADE_LAMBDA, noise=NOISE_LEVEL, seed=SEED):
    rng = np.random.RandomState(seed)
    t = np.arange(n, dtype=float)
    return np.clip(np.exp(-lam * t) + rng.normal(0, noise, n), 0, 1)

def gen_abrupt_cliff(n=TRAJECTORY_LENGTH, cliff_m=CLIFF_MONTH, noise=NOISE_LEVEL, seed=SEED):
    rng = np.random.RandomState(seed)
    s = np.ones(n) + rng.normal(0, noise, n)
    s[cliff_m:] = 0.05 + rng.normal(0, noise, n - cliff_m)
    return np.clip(s, 0, 1)

def gen_plateau_then_cliff(n=TRAJECTORY_LENGTH, break_m=16, noise=NOISE_LEVEL, seed=SEED):
    rng = np.random.RandomState(seed)
    pre = np.ones(break_m) + rng.normal(0, noise, break_m)
    post = np.linspace(0.9, 0.0, n - break_m) + rng.normal(0, noise, n - break_m)
    return np.clip(np.concatenate([pre, post]), 0, 1)

In [ ]:
# Generate and visualize synthetic trajectories
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
patterns = [
    ('smooth_fade', gen_smooth_fade, 'Smooth Fade'),
    ('abrupt_cliff', gen_abrupt_cliff, 'Abrupt Cliff'),
    ('plateau_then_cliff', gen_plateau_then_cliff, 'Plateau-Then-Cliff'),
]

for (pattern, gen_func, title), ax in zip(patterns, axes):
    traj = gen_func(seed=SEED)
    ax.plot(traj, 'o-', linewidth=2, markersize=6)
    ax.set_title(f'{title} Pattern', fontsize=12)
    ax.set_xlabel('Month')
    ax.set_ylabel('Founder Share')
    ax.set_xlim(0, TRAJECTORY_LENGTH - 1)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Phase 0: Trajectory Shape Descriptors

For each trajectory, we compute a set of **shape descriptors** that capture different aspects of the founder's involvement decline:
- `slope`: Linear trend coefficient
- `r2_linear`: Goodness of linear fit
- `normalized_slope`: Slope relative to mean involvement
- `quadratic_coef`: Convexity (curvature) of the trajectory
- `onset_decline_month`: When decline begins (change-point detection)
- `decline_duration_fraction`: How long the decline lasts
- `cliff_indicator`: Magnitude of the largest month-to-month change
- `fade_index`: Composite measure of fade-like behavior (normalized across sample)

In [ ]:
# Descriptor computation (from method.py)
def compute_all_descriptors(shares, label="unknown"):
    """Compute all trajectory shape descriptors from a monthly share array."""
    y = np.array(shares, dtype=float)
    n = len(y)
    
    if n < 3:
        return {k: 0.0 for k in [
            'slope', 'r2_linear', 'normalized_slope', 'quadratic_coef',
            'onset_decline_month', 'decline_duration_fraction',
            'cliff_indicator', 'cliff_is_terminal', 'plateau_then_cliff',
            'fade_index'
        ]}
    
    x = np.arange(n, dtype=float)
    res = {}

    # (a) LINEAR SLOPE via OLS
    slope = 0.0
    r2_linear = 0.0
    try:
        if np.all(y == y[0]):
            slope = 0.0
            r2_linear = 0.0
        else:
            coeffs = np.polyfit(x, y, 1)
            slope = float(coeffs[0])
            y_pred = np.polyval(coeffs, x)
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            r2_linear = float(1 - ss_res / ss_tot) if ss_tot > 1e-10 else 0.0
    except Exception as e:
        print(f"  slope computation failed: {e}")

    mean_share = max(float(np.mean(y)), 1e-8)
    res['slope'] = slope
    res['r2_linear'] = r2_linear
    res['normalized_slope'] = slope / mean_share

    # (b) CONVEXITY via quadratic fit
    try:
        coeffs = np.polyfit(x, y, 2)
        res['quadratic_coef'] = float(coeffs[0])
    except Exception:
        res['quadratic_coef'] = 0.0

    # (c) TIME-TO-ONSET-OF-DECLINE via sliding window F-statistic
    onset = 0
    best_f = -1
    best_split = 0
    for split in range(2, n - 1):
        pre = y[:split]
        post = y[split:]
        if len(pre) < 2 or len(post) < 2:
            continue
        var_pre = np.var(pre)
        var_post = np.var(post)
        mean_pre = np.mean(pre)
        mean_post = np.mean(post)
        if var_pre < 1e-10 and var_post < 1e-10:
            continue
        pooled_var = (var_pre + var_post) / 2
        if pooled_var < 1e-10:
            continue
        f_stat = ((mean_pre - mean_post) ** 2) / (pooled_var * (1/len(pre) + 1/len(post)))
        if f_stat > best_f:
            best_f = f_stat
            best_split = split
    onset = best_split

    res['onset_decline_month'] = onset
    res['decline_duration_fraction'] = float((n - onset) / n) if n > 0 else 0.0

    # (d) ABRUPT-CLIFF INDICATOR
    diffs = np.abs(np.diff(y))
    cliff_mag = float(np.max(diffs)) if len(diffs) > 0 else 0.0
    traj_std = float(np.std(y))
    mad = float(np.mean(diffs)) if len(diffs) > 0 else 1e-8
    if traj_std > 1e-10:
        cliff_ind = float(cliff_mag / (2 * traj_std + 1e-8))
    elif mad > 1e-10:
        cliff_ind = float(cliff_mag / (2 * mad + 1e-8))
    else:
        cliff_ind = 0.0
    cliff_month = int(np.argmax(diffs)) if len(diffs) > 0 else 0
    cliff_is_terminal = cliff_month >= n - 3
    res['cliff_indicator'] = cliff_ind
    res['cliff_is_terminal'] = cliff_is_terminal

    # (e) PLATEAU-THEN-CLIFF INDICATOR
    plateau_score = 0.0
    try:
        if onset > 2 and onset < n - 2:
            pre = y[:onset]
            post = y[onset:]
            if len(pre) >= 2 and len(post) >= 2:
                pre_x = np.arange(len(pre), dtype=float)
                post_x = np.arange(len(post), dtype=float)
                try:
                    pre_slope = float(np.polyfit(pre_x, pre, 1)[0])
                    post_slope = float(np.polyfit(post_x, post, 1)[0])
                except Exception:
                    pre_slope = 0.0
                    post_slope = 0.0
                pre_mean = float(np.mean(pre))
                if abs(pre_slope) < 0.02 and pre_mean > 0.5 and post_slope < -0.02:
                    plateau_score = 1.0
                elif abs(pre_slope) < 0.03 and pre_mean > 0.4 and post_slope < -0.01:
                    plateau_score = 0.6
                elif post_slope < -0.02:
                    plateau_score = 0.3
    except Exception:
        plateau_score = 0.0
    res['plateau_then_cliff'] = plateau_score

    # (f) Store raw components for batch normalization
    res['_slope_abs'] = abs(slope)
    res['_decline_dur'] = res['decline_duration_fraction']
    res['_cliff_mag_norm'] = cliff_ind
    res['fade_index'] = 0.0  # placeholder

    return res

def compute_fade_index_batch(all_descriptors):
    """Compute fade_index with min-max normalization across the sample."""
    if len(all_descriptors) < 2:
        for d in all_descriptors:
            d['fade_index'] = 0.5
        return all_descriptors

    slope_abs_vals = [d.get('_slope_abs', 0) for d in all_descriptors]
    decline_dur_vals = [d.get('_decline_dur', 0) for d in all_descriptors]
    cliff_vals = [d.get('_cliff_mag_norm', 0) for d in all_descriptors]

    def minmax(vals):
        mn, mx = min(vals), max(vals)
        if mx - mn < 1e-10:
            return [0.5] * len(vals)
        return [(v - mn) / (mx - mn) for v in vals]

    norm_slope = minmax(slope_abs_vals)
    norm_decline = minmax(decline_dur_vals)
    norm_cliff = minmax(cliff_vals)

    for i, d in enumerate(all_descriptors):
        fade = (0.3 * (1 - norm_slope[i]) +
                0.3 * norm_decline[i] +
                0.4 * (1 - norm_cliff[i]))
        d['fade_index'] = float(np.clip(fade, 0, 1))

    return all_descriptors

In [ ]:
# Compute descriptors for synthetic trajectories
all_synthetic = []
for i in range(N_TRAJECTORIES):
    traj = gen_smooth_fade(seed=SEED + i)
    desc = compute_all_descriptors(traj)
    all_synthetic.append(('smooth_fade', desc))

for i in range(N_TRAJECTORIES):
    traj = gen_abrupt_cliff(seed=SEED + i)
    desc = compute_all_descriptors(traj)
    all_synthetic.append(('abrupt_cliff', desc))

for i in range(N_TRAJECTORIES):
    traj = gen_plateau_then_cliff(seed=SEED + i)
    desc = compute_all_descriptors(traj)
    all_synthetic.append(('plateau_then_cliff', desc))

# Compute fade_index with batch normalization
all_desc = [item[1] for item in all_synthetic]
compute_fade_index_batch(all_desc)

# Print descriptor summary
print(f"Computed descriptors for {len(all_synthetic)} synthetic trajectories")
print("\nSample descriptors (first smooth_fade):")
for k, v in all_synthetic[0][1].items():
    if not k.startswith('_'):
        print(f"  {k}: {v:.4f}")

## Phase 0: Synthetic Validation

We validate our descriptors by checking assertions about the expected patterns:
- Smooth fades should have high fade_index (>0.5) and low cliff_indicator
- Abrupt cliffs should have low fade_index (<0.5) and high cliff_indicator
- Plateau-then-cliff should have high plateau_then_cliff score
- There should be separation between smooth_fade and abrupt_cliff fade_index

In [ ]:
# Aggregate stats per pattern
stats = {}
for pattern in ['smooth_fade', 'abrupt_cliff', 'plateau_then_cliff']:
    items = [item for item in all_synthetic if item[0] == pattern]
    desc_list = [item[1] for item in items]
    stats[pattern] = {
        'mean_fade_index': float(np.mean([d['fade_index'] for d in desc_list])),
        'mean_cliff_indicator': float(np.mean([d['cliff_indicator'] for d in desc_list])),
        'mean_decline_duration': float(np.mean([d['decline_duration_fraction'] for d in desc_list])),
        'mean_plateau_then_cliff': float(np.mean([d['plateau_then_cliff'] for d in desc_list])),
        'mean_slope': float(np.mean([d['slope'] for d in desc_list])),
    }

# Assertions
assertions = {}
sf = stats['smooth_fade']
ac = stats['abrupt_cliff']
pc = stats['plateau_then_cliff']

assertions['smooth_fade_fade_index_gt_0.5'] = sf['mean_fade_index'] > 0.5
assertions['smooth_fade_cliff_lt_2.5'] = sf['mean_cliff_indicator'] < 2.5
assertions['smooth_fade_decline_gt_0.4'] = sf['mean_decline_duration'] > 0.4
assertions['abrupt_cliff_fade_index_lt_0.5'] = ac['mean_fade_index'] < 0.5
assertions['abrupt_cliff_cliff_gt_0.5'] = ac['mean_cliff_indicator'] > 0.5
assertions['plateau_cliff_plateau_indicator_gt_0.3'] = pc['mean_plateau_then_cliff'] > 0.3
assertions['fade_index_separation'] = sf['mean_fade_index'] > ac['mean_fade_index']

passed = sum(1 for v in assertions.values() if v)
total = len(assertions)

print(f"Synthetic validation: {passed}/{total} assertions passed")
for name, val in assertions.items():
    status = "PASS" if val else "FAIL"
    print(f"  [{status}] {name}")

## Load Pre-computed Results

The full experiment analyzed 14 curated GitHub repos with documented founder departures (7 survived, 7 collapsed). Let's load and visualize the pre-computed results.

In [ ]:
# Parse examples from loaded data
synthetic_examples = [ex for ex in examples if ex.get('metadata_pattern')]
project_examples = [ex for ex in examples if ex.get('metadata_repo')]
model_examples = [ex for ex in examples if ex.get('metadata_model')]
validation_example = [ex for ex in examples if 'Synthetic trajectory validation' in ex.get('input', '')]
falsification_example = [ex for ex in examples if 'Falsification control' in ex.get('input', '')]

print(f"Synthetic trajectories: {len(synthetic_examples)}")
print(f"Projects analyzed: {len(project_examples)}")
print(f"Model comparisons: {len(model_examples)}")
print(f"Validation summary: {len(validation_example)}")
print(f"Falsification control: {len(falsification_example)}")

In [ ]:
# Parse validation results
if validation_example:
    val_data = json.loads(validation_example[0]['output'])
    print("=== Synthetic Validation Results ===")
    print(f"Passed: {val_data['passed']}/{val_data['total']}")
    print("\nPattern Statistics:")
    for pattern, stats in val_data['stats'].items():
        print(f"\n{pattern}:")
        for k, v in stats.items():
            print(f"  {k}: {v:.4f}")
    print("\nAssertions:")
    for name, passed in val_data['assertions'].items():
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")

## Project Analysis Results

The experiment analyzed 14 real GitHub projects with documented founder departures. Here are the key findings for each project:

In [ ]:
# Parse and display project results
project_data = []
for ex in project_examples:
    out = json.loads(ex['output'])
    project_data.append({
        'repo': out['repo'],
        'founder': out['founder'],
        'survival_label': out['survival_label'],
        'expected_survival': out['expected_survival'],
        'survival_ratio': out['survival_ratio'],
        'fade_index': out['fade_index'],
        'cliff_indicator': out['cliff_indicator'],
        'slope': out['slope'],
        'r2_linear': out['r2_linear'],
    })

# Display as table
df_projects = pd.DataFrame(project_data)
df_projects['survived'] = df_projects['survival_label'].map({1: '✓', 0: '✗'})
df_projects_display = df_projects[['repo', 'survived', 'expected_survival', 'survival_ratio', 'fade_index', 'cliff_indicator']]
print(df_projects_display.to_string(index=False))

## Model Comparison Results

We trained three logistic regression models with Leave-One-Out Cross-Validation (LOOCV):
1. **Static features only**: contributor_count, total_commits, bus_factor, etc.
2. **Shape descriptors only**: slope, cliff_indicator, fade_index, etc.
3. **Combined**: Both static and shape features

Key finding: Static features achieve AUC=0.857, but shape descriptors alone (AUC=0.408) perform below chance.

In [ ]:
# Parse model comparison results
model_data = {}
for ex in model_examples:
    out = json.loads(ex['output'])
    model_data[out['model']] = {
        'AUC': out['loocv_auc'],
        'accuracy': out['loocv_accuracy'],
        'feature_importance': out['feature_importance'],
    }

# Display model comparison
print("=== Model Comparison (LOOCV) ===")
for model_name, metrics in model_data.items():
    print(f"\n{model_name}:")
    print(f"  AUC: {metrics['AUC']:.3f}")
    print(f"  Accuracy: {metrics['accuracy']:.3f}")
    print(f"  Top features:")
    sorted_features = sorted(metrics['feature_importance'].items(), 
                             key=lambda x: abs(x[1]), reverse=True)[:3]
    for feat, imp in sorted_features:
        print(f"    {feat}: {imp:.4f}")

## Falsification Control

To test whether founder-specific effects exist, we ran a falsification control using non-founder trajectories. If founder fade curves are meaningful, they should predict survival better than non-founder trajectories.

In [ ]:
# Parse falsification control results
if falsification_example:
    fc_data = json.loads(falsification_example[0]['output'])
    print("=== Falsification Control ===")
    print(f"Founder AUC: {fc_data['founder_auc']:.3f}")
    print(f"Non-founder AUC: {fc_data['non_founder_auc']:.3f}")
    print(f"Delta: {fc_data['delta']:.3f}")
    print(f"Founder-specific effect: {fc_data['founder_specific']}")
    print("\nConclusion: No founder-specific effect detected (founder_AUC == non_founder_AUC)")

## Visualization: Key Results

Let's visualize the main findings from the experiment.

In [ ]:
# Create visualization figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Synthetic trajectory patterns
ax1 = axes[0, 0]
for pattern, gen_func, color in [
    ('Smooth Fade', gen_smooth_fade, 'green'),
    ('Abrupt Cliff', gen_abrupt_cliff, 'red'),
    ('Plateau-Then-Cliff', gen_plateau_then_cliff, 'orange')
]:
    traj = gen_func(seed=SEED)
    ax1.plot(traj, 'o-', color=color, label=pattern, markersize=6, alpha=0.7)
ax1.set_title('Synthetic Trajectory Patterns', fontsize=12)
ax1.set_xlabel('Month')
ax1.set_ylabel('Founder Share')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)

# 2. Model comparison AUC
ax2 = axes[0, 1]
models = list(model_data.keys())
auc_values = [model_data[m]['AUC'] for m in models]
colors = ['steelblue', 'coral', 'seagreen']
bars = ax2.bar(models, auc_values, color=colors, edgecolor='black', linewidth=0.5)
ax2.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Chance (0.5)')
ax2.set_title('Model Comparison (LOOCV AUC)', fontsize=12)
ax2.set_ylabel('AUC')
ax2.set_ylim(0, 1.1)
ax2.legend()
# Add value labels on bars
for bar, val in zip(bars, auc_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Fade index vs survival
ax3 = axes[1, 0]
survived = [p for p in project_data if p['survival_label'] == 1]
collapsed = [p for p in project_data if p['survival_label'] == 0]
ax3.scatter([p['fade_index'] for p in survived], 
            [1]*len(survived), color='green', s=100, alpha=0.6, label='Survived', zorder=5)
ax3.scatter([p['fade_index'] for p in collapsed], 
            [0]*len(collapsed), color='red', s=100, alpha=0.6, label='Collapsed', zorder=5)
ax3.set_title('Fade Index vs Survival Outcome', fontsize=12)
ax3.set_xlabel('Fade Index')
ax3.set_ylabel('Survival (1=Survived, 0=Collapsed)')
ax3.set_ylim(-0.1, 1.1)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='x')

# 4. Project survival rates
ax4 = axes[1, 1]
survival_counts = df_projects['expected_survival'].value_counts().sort_index()
survival_labels = [f'Collapsed (0)' if k == 0 else f'Survived (1)' for k in survival_counts.index]
colors_pie = ['coral', 'seagreen']
ax4.pie(survival_counts.values, labels=survival_labels, colors=colors_pie, 
        autopct='%1.0f%%', startangle=90)
ax4.set_title('Project Survival Distribution', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# Fix the pie chart syntax error
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3. Fade index vs survival
ax3 = axes[0]
survived = [p for p in project_data if p['survival_label'] == 1]
collapsed = [p for p in project_data if p['survival_label'] == 0]
ax3.scatter([p['fade_index'] for p in survived], 
            [1]*len(survived), color='green', s=100, alpha=0.6, label='Survived', zorder=5)
ax3.scatter([p['fade_index'] for p in collapsed], 
            [0]*len(collapsed), color='red', s=100, alpha=0.6, label='Collapsed', zorder=5)
ax3.set_title('Fade Index vs Survival Outcome', fontsize=12)
ax3.set_xlabel('Fade Index')
ax3.set_ylabel('Survival (1=Survived, 0=Collapsed)')
ax3.set_ylim(-0.1, 1.1)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='x')

# 4. Project survival rates
ax4 = axes[1]
survival_counts = df_projects['expected_survival'].value_counts().sort_index()
survival_labels = ['Collapsed (0)', 'Survived (1)']
colors_pie = ['coral', 'seagreen']
ax4.pie(survival_counts.values, labels=survival_labels, colors=colors_pie, 
        autopct='%1.0f%%', startangle=90)
ax4.set_title('Project Survival Distribution', fontsize=12)

plt.tight_layout()
plt.show()

## Summary of Key Findings

1. **Synthetic validation**: All 7 assertions passed, confirming descriptors correctly identify fade/cliff patterns across synthetic trajectories.

2. **Static features are predictive**: AUC=0.857 via LOOCV logistic regression using contributor_count, total_commits, bus_factor, etc.

3. **Trajectory shape descriptors alone are NOT predictive**: AUC=0.408 (below chance), suggesting fade_index and related metrics don't significantly predict survival on their own.

4. **Combined features perform best**: AUC=0.898 with CoxPH concordance=0.92.

5. **No founder-specific effect**: Falsification control found founder_AUC = non_founder_AUC = 0.41, suggesting the fade curve pattern is not unique to founders.

**Conclusion**: In this pilot study, trajectory shape descriptors do not significantly predict survival beyond static features. The most predictive factors are structural (contributor count, total commits, bus factor) rather than temporal (fade curve shape).

In [ ]:
# Final summary table
print("=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"\nSynthetic Validation: {val_data['passed']}/{val_data['total']} assertions passed")
print(f"Projects Analyzed: {len(project_examples)} (7 survived, 7 collapsed)")
print(f"\nModel Performance (LOOCV):")
for model_name, metrics in model_data.items():
    print(f"  {model_name:20s} AUC={metrics['AUC']:.3f}  Accuracy={metrics['accuracy']:.3f}")
if falsification_example:
    print(f"\nFalsification Control:")
    print(f"  Founder AUC: {fc_data['founder_auc']:.3f}")
    print(f"  Non-founder AUC: {fc_data['non_founder_auc']:.3f}")
    print(f"  Founder-specific: {fc_data['founder_specific']}")
print("\n" + "=" * 70)